# Generalización cross-age en biometría de oreja: evaluación entre población adulta e infantil

**Autor:** Rafael Suárez Saavedra

**Tutor:** David Sebastián Freire Obregón

**Grado:** Ciencia e Ingeniería de Datos  

**Universidad:** ULPGC  

---
Este notebook contiene el código utilizado para los experimentos del TFG.

El objetivo es evaluar la capacidad de generalización de distintos modelos de Deep Learning entrenados con imágenes de orejas adultas y evaluados sobre población infantil.

---

## Índice

1. [Instrucciones de uso](#1-instrucciones-de-uso)
2. [Preprocesado de datos](#2-preprocesado-de-datos)
3. [Desarrollo del modelo](#3-desarrollo-del-modelo)
4. [Experimentos base](#4-experimentos-base)
5. [Triplet Loss](#4-triplet-loss)
6. [Ajuste de hiperparámetros](#5-tuning-de-hiperparámetros)
7. [Comparación de backbones](#6-comparación-de-backbones)
8. [Fusión de embeddings](#7-fusión-de-embeddings)


## 1. Instrucciones de uso

Para ejecutar este notebook correctamente se recomienda utilizar Google Colab con la GPU activada.

Pasos recomendados:

1. Abrir el notebook en Google Colab.
2. Activar la GPU desde `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.
3. Ejecutar las celdas de las importaciones y descarga de los datos y el código del apartado, [Preprocesado de Datos](#2-preprocesado-de-datos) y [Desarrollo del modelo](#3-desarrollo-del-modelo). Posteriormente, ejecutar el experimento que se desee realizar. También puede ejecutarse el notebook completo si se quieren reproducir todos los experimentos.
4. Para modificar o comparar distintos experimentos, se deben ajustar los parámetros situados al comienzo de cada bloque experimental.

Los resultados generados se almacenan en la carpeta `runs/`. Esta carpeta incluye las métricas en formato CSV, las gráficas comparativas y los archivos asociados a cada ejecución. Además, el archivo `all_experiments_results.csv` recoge un resumen comparativo de los experimentos realizados.

El tiempo aproximado de ejecución del notebook completo es de 40 minutos.

In [ ]:
import os
import shutil
import pandas as pd
import re
from pathlib import Path
import random
import itertools
import numpy as np
from PIL import Image
from tqdm import tqdm
import gdown
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms, models
from sklearn.metrics import roc_curve, roc_auc_score
import torch.nn.functional as F
import json
import matplotlib.pyplot as plt
import tempfile
import keras


In [ ]:
url = "https://drive.google.com/file/d/1Z25W_qKx71QmhKOB0oQcksrgCAamBSYH/view?usp=drive_link"
output = "TFG_dataset.zip"

gdown.download(url, output, quiet=False, fuzzy=True)

In [ ]:
!unzip -q /content/TFG_dataset.zip -d /content
!rm /content/TFG_dataset.zip

##2. Preprocesado de datos



In [ ]:
#Preprocesado AMI

source_dir = "Datos_TFG_Rafael"
OUT_CSV = "individual_metadata/metadata_AMI.csv"
dest_dir = "images"
ami_dir = source_dir + "/AMI"

os.makedirs(dest_dir, exist_ok=True)
os.makedirs("individual_metadata", exist_ok=True)

metadata = []

for file in os.listdir(ami_dir):
    if file.endswith(".jpg"):
        parts = file.split("_")
        subject = parts[0]
        pose = parts[1]

        new_name = f"AMI_{subject}_{pose}.jpg"

        shutil.copy(
            os.path.join(ami_dir, file),
            os.path.join(dest_dir, new_name)
        )

        metadata.append({
            "image_id": new_name.replace(".jpg",""),
            "subject_id": f"AMI_{subject}",
            "dataset": "AMI",
            "age_group": "adult",
            "image_path": f"images/{new_name}",
            "pose": pose
        })

df = pd.DataFrame(metadata)
df.to_csv(OUT_CSV, index=False)


In [ ]:
#Preprocesado BIPLab

OUT_CSV =  "individual_metadata/metadata_BIPLab.csv"
biplab_dir= source_dir + "/BIPLab/Ear"
os.makedirs(dest_dir, exist_ok=True)

pattern = re.compile(r"^(ID\d+)_([A-Z]{2})_SAMPLE(\d+)\.(bmp|png|jpg|jpeg)$", re.IGNORECASE)

rows = []
skipped = []

for fname in os.listdir(biplab_dir):
    fpath = os.path.join(biplab_dir, fname)

    if not os.path.isfile(fpath):
        continue

    m = pattern.match(fname)
    if not m:
        skipped.append(fname)
        continue

    subj_raw = m.group(1).upper()
    side_code = m.group(2).upper()
    sample_id = m.group(3).zfill(3)
    ext = m.group(4).lower()


    ear_side = "left" if side_code == "SX" else ("right" if side_code == "DX" else "unknown")


    new_name = f"BIPLab_{subj_raw}_{sample_id}.{ext}"

    shutil.copy2(fpath, os.path.join(dest_dir, new_name))

    rows.append({
        "image_id": new_name.rsplit(".", 1)[0],
        "subject_id": f"BIPLab_{subj_raw}",
        "dataset": "BIPLab",
        "age_group": "adult",
        "image_path": f"images/{new_name}",
        "ear_side": ear_side,
        "sample_id": int(sample_id),
        "original_filename": fname,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

In [ ]:
#Preprocesado UERC

uerc_dir = source_dir + "/UERC/Dataset/Train Dataset"
OUT_CSV = "individual_metadata/metadata_UERC_train.csv"

os.makedirs(dest_dir, exist_ok=True)

rows = []
skipped = []

def iter_images(subject_dir: str):
    """Devuelve lista de ficheros imagen dentro de un directorio (no recursivo)."""
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    for f in os.listdir(subject_dir):
        if f.lower().endswith(exts) and os.path.isfile(os.path.join(subject_dir, f)):
            yield f

subject_folders = sorted([
    d for d in os.listdir(uerc_dir)
    if os.path.isdir(os.path.join(uerc_dir, d))
])

for subj in subject_folders:
    subj_path = os.path.join(uerc_dir, subj)

    img_files = sorted(list(iter_images(subj_path)))

    if len(img_files) == 0:
        skipped.append((subj, "no_images"))
        continue

    for idx, fname in enumerate(img_files, start=1):
        src = os.path.join(subj_path, fname)
        ext = os.path.splitext(fname)[1].lower()

        new_name = f"UERC_{subj}_{idx:02d}{ext}"

        dst = os.path.join(dest_dir, new_name)
        shutil.copy2(src, dst)

        rows.append({
            "image_id": new_name.rsplit(".", 1)[0],
            "subject_id": f"UERC_{subj}",
            "dataset": "UERC",
            "age_group": "adult",
            "image_path": f"images/{new_name}",
            "ear_side": "unknown",
            "original_filename": fname,
            "original_relpath": f"Train Dataset/{subj}/{fname}",
        })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)


In [ ]:
#Preprocesado EICZA

eicza_dir = source_dir + "/dataset_EICZA/jpgs"
OUT_CSV = "individual_metadata/metadata_EICZA.csv"

os.makedirs(dest_dir, exist_ok=True)


pattern = re.compile(
    r"^small_(.+?)_([0-9]+)([MWD])_(Left|Right)_(True|False)_Cropped\.(jpg|jpeg|png)$",
    re.IGNORECASE
)

rows = []
skipped = []

def normalize_subject(raw: str) -> str:
    s = raw.strip()
    s = re.sub(r"\s+copy$", "", s, flags=re.IGNORECASE)
    s = s.replace(" ", "")
    return s

def age_to_days(value: int, unit: str) -> float:
    unit = unit.upper()
    if unit == "D":
        return float(value)
    if unit == "W":
        return float(value) * 7.0
    if unit == "M":
        # 1 mes ≈ 30.437 días (promedio)
        return float(value) * 30.437
    raise ValueError("Unidad desconocida")

for fname in sorted(os.listdir(eicza_dir)):
    src_path = os.path.join(eicza_dir, fname)
    if not os.path.isfile(src_path):
        continue

    m = pattern.match(fname)
    if not m:
        skipped.append(fname)
        continue

    subj_raw = m.group(1)
    age_value = int(m.group(2))
    age_unit = m.group(3).upper()
    side_raw = m.group(4).lower()
    tf_raw = m.group(5).lower()
    ext = m.group(6).lower()

    subj = normalize_subject(subj_raw)

    ear_side = "left" if side_raw == "left" else "right"
    is_true = 1 if tf_raw == "true" else 0

    age_days = age_to_days(age_value, age_unit)
    age_months = round(age_days / 30.437, 2)

    side_code = "L" if ear_side == "left" else "R"
    tf_code = "T" if is_true == 1 else "F"

    new_name = f"EICZA_{subj}_{age_value}{age_unit}_{side_code}_{tf_code}.{ext}"
    dst_path = os.path.join(dest_dir, new_name)

    shutil.copy2(src_path, dst_path)

    rows.append({
        "image_id": new_name.rsplit(".", 1)[0],
        "subject_id": f"EICZA_{subj}",
        "dataset": "EICZA",
        "age_group": "child",
        "age_value": age_value,
        "age_unit": age_unit,
        "age_days": round(age_days, 2),
        "age_months": age_months,
        "ear_side": ear_side,
        "is_true_crop": is_true,
        "rotation_flag": "",
        "image_path": f"images/{new_name}",
        "original_filename": fname,
        "quality_code": ""
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)


In [ ]:
#Puesta en común en un mismo csv

def norm_col(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def load_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [norm_col(c) for c in df.columns]
    return df


def map_age_group(x):
    if pd.isna(x):
        return x
    s = str(x).strip().lower()
    mapping = {
        "adult": "adulto",
        "adulto": "adulto",
        "child": "niño",
        "infant": "niño",
        "kid": "niño",
        "nino": "niño",
        "niño": "niño",
    }
    return mapping.get(s, x)


def coalesce_cols(df: pd.DataFrame, candidates: list[str]):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def norm_ear_side(value):
    if pd.isna(value):
        return pd.NA
    s = str(value).strip().lower()

    mapping = {
        # inglés
        "left": "left",
        "right": "right",
        "l": "left",
        "r": "right",
        # español
        "izquierda": "left",
        "derecha": "right",
        "izq": "left",
        "der": "right",
        # codificaciones típicas
        "sx": "left",
        "dx": "right",
    }
    if s in mapping:
        return mapping[s]

    if "left" in s or "_l" in s or " sx" in s or "_sx" in s:
        return "left"
    if "right" in s or "_r" in s or " dx" in s or "_dx" in s:
        return "right"

    return pd.NA


def norm_age_unit(u):
    if pd.isna(u):
        return pd.NA
    s = str(u).strip().lower()
    mapping = {
        "day": "days", "days": "days", "d": "days",
        "week": "weeks", "weeks": "weeks", "w": "weeks",
        "month": "months", "months": "months", "m": "months",
        "year": "years", "years": "years", "y": "years",
    }
    return mapping.get(s, s)


def infer_ear_side_ami(df: pd.DataFrame) -> pd.Series:
    if "pose" not in df.columns:
        return pd.Series([pd.NA] * len(df), index=df.index)
    pose = df["pose"].astype(str).str.strip().str.lower()
    return pose.apply(lambda p: "left" if p == "back" else "right")


def infer_ear_side_from_existing(df: pd.DataFrame) -> pd.Series:
    col = coalesce_cols(df, ["ear_side", "side", "ear", "laterality"])
    if col is None:
        return pd.Series([pd.NA] * len(df), index=df.index)
    return df[col].apply(norm_ear_side)



def build_unified_metadata(
    path_ami: str | Path,
    path_biplab: str | Path,
    path_eicza: str | Path,
    path_uerc: str | Path,
    output_path: str | Path = "metadata_unificado_final.csv",
) -> pd.DataFrame:
    ami = load_csv(path_ami)
    biplab = load_csv(path_biplab)
    eicza = load_csv(path_eicza)
    uerc = load_csv(path_uerc)

    datasets = [
        ("AMI", ami),
        ("BIPLab", biplab),
        ("EICZA", eicza),
        ("UERC", uerc),
    ]

    out_frames = []

    for name, df in datasets:
        out = pd.DataFrame()

        out["image_id"] = df["image_id"] if "image_id" in df.columns else pd.NA
        out["subject_id"] = df["subject_id"] if "subject_id" in df.columns else pd.NA
        out["dataset"] = df["dataset"] if "dataset" in df.columns else name
        out["age_group"] = df["age_group"].apply(map_age_group) if "age_group" in df.columns else pd.NA
        out["image_path"] = df["image_path"] if "image_path" in df.columns else pd.NA

        if name == "AMI":
            out["ear_side"] = infer_ear_side_ami(df)
        elif name in ("BIPLab", "EICZA"):
            out["ear_side"] = infer_ear_side_from_existing(df)

        if name == "EICZA":
            col_age_value = coalesce_cols(df, ["age_value", "age", "edad"])
            col_age_unit = coalesce_cols(df, ["age_unit", "unit", "age_units", "edad_unit"])

            out["age_value"] = pd.to_numeric(df[col_age_value], errors="coerce") if col_age_value else pd.NA
            out["age_unit"] = df[col_age_unit].apply(norm_age_unit) if col_age_unit else pd.NA
        else:
            out["age_value"] = -1
            out["age_unit"] = "none"



        out_frames.append(out)

    merged = pd.concat(out_frames, ignore_index=True)

    final_cols = [
        "image_id",
        "subject_id",
        "dataset",
        "age_group",
        "image_path",
        "ear_side",
        "age_value",
        "age_unit",
    ]
    merged = merged[final_cols]

    merged.to_csv(output_path, index=False)
    return merged



df = build_unified_metadata(
    path_ami= "individual_metadata/metadata_AMI.csv",
    path_biplab= "individual_metadata/metadata_BIPLab.csv",
    path_eicza= "individual_metadata/metadata_EICZA.csv",
    path_uerc= "individual_metadata/metadata_UERC_train.csv",
    output_path= "metadata_unificado_final.csv",
)


##Desarrollo del modelo

In [ ]:
METADATA_CSV = "metadata_unificado_final.csv"
IMAGES_ROOT = "/content/"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_dir(path):
    os.makedirs(path, exist_ok=True)


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def normalize_embeddings(x):
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / (norm + 1e-12)


def cosine_similarity(a, b):
    return float(np.dot(a, b))


def compute_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer), fpr, tpr


def filter_by_datasets(df, datasets):
    return df[df["dataset"].isin(datasets)].copy()


def split_train_val_by_subject(df, val_ratio=0.1, seed=42):
    subjects = df["subject_id"].astype(str).unique().tolist()

    rng = np.random.RandomState(seed)
    rng.shuffle(subjects)

    n_val = max(1, int(len(subjects) * val_ratio))
    val_subjects = set(subjects[:n_val])
    train_subjects = set(subjects[n_val:])

    df_train = df[df["subject_id"].astype(str).isin(train_subjects)].copy()
    df_val = df[df["subject_id"].astype(str).isin(val_subjects)].copy()

    return df_train, df_val

In [ ]:
class EarDataset(Dataset):
    def __init__(self, df, images_root, transform=None, label_column=None):
        self.df = df.reset_index(drop=True)
        self.images_root = images_root
        self.transform = transform
        self.label_column = label_column

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.images_root, str(row["image_path"]))
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.label_column is None:
            return image

        label = int(row[self.label_column])
        return image, label

In [ ]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, num_classes, embedding_dim=512):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embedding_dim, embedding_dim)
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        emb = self.embedding(features)
        logits = self.classifier(emb)
        return logits, emb

In [ ]:
def batch_accuracy(logits, y):
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    all_losses = []
    all_accs = []

    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        all_losses.append(loss.item())
        all_accs.append(batch_accuracy(logits, y))

    return float(np.mean(all_losses)), float(np.mean(all_accs))

In [ ]:
@torch.no_grad()
def extract_embeddings(model, df, images_root, img_size, batch_size, num_workers, device):
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = EarDataset(df, images_root, transform=transform, label_column=None)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    embeddings = []

    for x in tqdm(loader, desc="embeddings", leave=False):
        x = x.to(device, non_blocking=True)
        _, emb = model(x)
        embeddings.append(emb.cpu().numpy())

    embeddings = np.vstack(embeddings)
    embeddings = normalize_embeddings(embeddings)

    return embeddings

In [ ]:
def sample_verification_pairs(embeddings, subject_ids, n_genuine, n_impostor, seed=42):
    rng = np.random.RandomState(seed)

    indices_by_subject = {}
    for i, s in enumerate(subject_ids):
        indices_by_subject.setdefault(s, []).append(i)

    subjects = list(indices_by_subject.keys())
    valid_subjects = [s for s in subjects if len(indices_by_subject[s]) >= 2]

    if len(valid_subjects) == 0:
        raise ValueError("No hay sujetos con al menos 2 imágenes para generar pares genuinos.")

    if len(subjects) < 2:
        raise ValueError("Se necesitan al menos 2 sujetos para generar pares impostores.")

    genuine_scores = []
    impostor_scores = []

    for _ in range(n_genuine):
        s = rng.choice(valid_subjects)
        i1, i2 = rng.choice(indices_by_subject[s], size=2, replace=False)
        score = cosine_similarity(embeddings[i1], embeddings[i2])
        genuine_scores.append(score)

    for _ in range(n_impostor):
        s1, s2 = rng.choice(subjects, size=2, replace=False)
        i1 = rng.choice(indices_by_subject[s1])
        i2 = rng.choice(indices_by_subject[s2])
        score = cosine_similarity(embeddings[i1], embeddings[i2])
        impostor_scores.append(score)

    y_true = np.array([1] * len(genuine_scores) + [0] * len(impostor_scores))
    y_score = np.array(genuine_scores + impostor_scores)

    return y_true, y_score

In [ ]:
def split_gallery_probe_by_subject(df, seed=42):
    rng = np.random.RandomState(seed)

    gallery_rows = []
    probe_rows = []

    grouped = df.groupby(df["subject_id"].astype(str))

    for subject_id, group in grouped:
        group = group.sample(frac=1.0, random_state=rng.randint(0, 10_000)).reset_index(drop=True)

        if len(group) >= 2:
            gallery_rows.append(group.iloc[[0]])
            probe_rows.append(group.iloc[1:])
        else:
            gallery_rows.append(group.iloc[[0]])

    df_gallery = pd.concat(gallery_rows, axis=0).reset_index(drop=True)
    if len(probe_rows) > 0:
        df_probe = pd.concat(probe_rows, axis=0).reset_index(drop=True)
    else:
        df_probe = pd.DataFrame(columns=df.columns)

    return df_gallery, df_probe


def compute_rank_k(gallery_embeddings, gallery_subject_ids, probe_embeddings, probe_subject_ids, k=1):
    if len(probe_embeddings) == 0:
        return np.nan

    sims = probe_embeddings @ gallery_embeddings.T
    topk_idx = np.argsort(-sims, axis=1)[:, :k]

    correct = 0
    for i in range(len(probe_subject_ids)):
        retrieved_subjects = gallery_subject_ids[topk_idx[i]]
        if probe_subject_ids[i] in retrieved_subjects:
            correct += 1

    return float(correct / len(probe_subject_ids))

In [ ]:
@torch.no_grad()
def evaluate_open_set(model, df_eval, images_root, img_size, batch_size, num_workers, device,
                      n_genuine_pairs=3000, n_impostor_pairs=3000, seed=42):
    if len(df_eval) == 0:
        raise ValueError("El dataframe de evaluación está vacío.")


    embeddings = extract_embeddings(
        model, df_eval, images_root, img_size, batch_size, num_workers, device
    )
    subject_ids = df_eval["subject_id"].astype(str).values

    y_true, y_score = sample_verification_pairs(
        embeddings, subject_ids, n_genuine_pairs, n_impostor_pairs, seed=seed
    )

    roc_auc = roc_auc_score(y_true, y_score)
    eer, fpr, tpr = compute_eer(y_true, y_score)


    df_gallery, df_probe = split_gallery_probe_by_subject(df_eval, seed=seed)

    if len(df_probe) == 0:
        rank1 = np.nan
        rank5 = np.nan
    else:
        gallery_embeddings = extract_embeddings(
            model, df_gallery, images_root, img_size, batch_size, num_workers, device
        )
        probe_embeddings = extract_embeddings(
            model, df_probe, images_root, img_size, batch_size, num_workers, device
        )

        gallery_subject_ids = df_gallery["subject_id"].astype(str).values
        probe_subject_ids = df_probe["subject_id"].astype(str).values

        rank1 = compute_rank_k(
            gallery_embeddings, gallery_subject_ids,
            probe_embeddings, probe_subject_ids,
            k=1
        )

        k5 = min(5, len(np.unique(gallery_subject_ids)))
        rank5 = compute_rank_k(
            gallery_embeddings, gallery_subject_ids,
            probe_embeddings, probe_subject_ids,
            k=k5
        )

    metrics = {
        "roc_auc": float(roc_auc),
        "eer": float(eer),
        "rank1": float(rank1) if not np.isnan(rank1) else np.nan,
        "rank5": float(rank5) if not np.isnan(rank5) else np.nan,
        "fpr": fpr,
        "tpr": tpr,
    }

    return metrics

In [ ]:
def prepare_train_val_data(df):

    df_train_all = df[df["age_group"] == TRAIN_AGE_GROUP].copy()
    df_train_all = filter_by_datasets(df_train_all, TRAIN_DATASETS)

    if len(df_train_all) == 0:
        raise ValueError("No hay datos de entrenamiento después del filtrado.")

    df_train, df_val = split_train_val_by_subject(
        df_train_all,
        VAL_RATIO,
        SEED
    )

    if len(df_train) == 0:
        raise ValueError("El conjunto de train quedó vacío.")

    if len(df_val) == 0:
        raise ValueError("El conjunto de validación quedó vacío.")

    train_subjects = sorted(df_train["subject_id"].astype(str).unique())
    subject_to_label = {s: i for i, s in enumerate(train_subjects)}

    df_train = df_train.copy()
    df_train["label"] = df_train["subject_id"].astype(str).map(subject_to_label)

    print("\n===== TRAIN / VAL INFO =====")
    print("Train datasets:", TRAIN_DATASETS)
    print("Test datasets:", TEST_DATASETS)
    print("Train subjects:", df_train["subject_id"].nunique())
    print("Val subjects:", df_val["subject_id"].nunique())
    print("Train images:", len(df_train))
    print("Val images:", len(df_val))

    return df_train, df_val, train_subjects

In [ ]:
def create_train_loader(df_train):

    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    train_dataset = EarDataset(
        df_train,
        IMAGES_ROOT,
        transform=train_transform,
        label_column="label"
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader

In [ ]:
def create_model(num_classes, device):

    model = EmbeddingClassifier(
        num_classes=num_classes,
        embedding_dim=EMBEDDING_DIM
    ).to(device)

    return model

In [ ]:
def initialize_history():

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_roc_auc": [],
        "val_eer": [],
        "val_rank1": [],
        "val_rank5": [],
    }

    return history

In [ ]:
def update_history(history, epoch, train_loss, train_acc, val_metrics):

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_roc_auc"].append(val_metrics["roc_auc"])
    history["val_eer"].append(val_metrics["eer"])
    history["val_rank1"].append(val_metrics["rank1"])
    history["val_rank5"].append(val_metrics["rank5"])

In [ ]:
def train_model(model, train_loader, df_val, device, best_model_path):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_val_eer = np.inf
    history = initialize_history()

    print("\n===== TRAINING =====")

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        val_metrics = evaluate_open_set(
            model=model,
            df_eval=df_val,
            images_root=IMAGES_ROOT,
            img_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            device=device,
            n_genuine_pairs=N_GENUINE_PAIRS,
            n_impostor_pairs=N_IMPOSTOR_PAIRS,
            seed=SEED
        )

        print(
            f"Epoch {epoch+1:02d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val AUC {val_metrics['roc_auc']:.4f} | "
            f"val EER {val_metrics['eer']:.4f} | "
            f"val R1 {val_metrics['rank1']:.4f} | "
            f"val R5 {val_metrics['rank5']:.4f}"
        )

        if val_metrics["eer"] < best_val_eer:
            best_val_eer = val_metrics["eer"]
            torch.save(model.state_dict(), best_model_path)
            print("  ✔ Modelo guardado en:", best_model_path)

        update_history(
            history=history,
            epoch=epoch,
            train_loss=train_loss,
            train_acc=train_acc,
            val_metrics=val_metrics
        )

    print("\nTraining finished.")
    print("Best val EER:", best_val_eer)

    return history, best_val_eer

In [ ]:
def prepare_test_data(df):

    df_test = df[df["age_group"] == TEST_AGE_GROUP].copy()
    df_test = filter_by_datasets(df_test, TEST_DATASETS)

    if len(df_test) == 0:
        raise ValueError("No hay datos de test después del filtrado.")

    print("\n===== TEST INFO =====")
    print("Test images:", len(df_test))
    print("Test subjects:", df_test["subject_id"].nunique())

    return df_test

In [ ]:
def evaluate_test_model(model, df_test, device):

    test_metrics = evaluate_open_set(
        model=model,
        df_eval=df_test,
        images_root=IMAGES_ROOT,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
        n_genuine_pairs=N_GENUINE_PAIRS,
        n_impostor_pairs=N_IMPOSTOR_PAIRS,
        seed=SEED
    )

    print("\n===== RESULTADOS TEST =====")
    print("ROC-AUC:", round(test_metrics["roc_auc"], 4))
    print("EER:", round(test_metrics["eer"], 4))
    print("Rank-1:", round(test_metrics["rank1"], 4))
    print("Rank-5:", round(test_metrics["rank5"], 4))

    return test_metrics

In [ ]:
def save_results_txt(test_metrics, best_val_eer):

    results_path = os.path.join(OUTPUT_DIR, "results.txt")

    with open(results_path, "w", encoding="utf-8") as f:
        f.write("Experiment name: " + str(EXPERIMENT_NAME) + "\n")
        f.write("Train datasets: " + str(TRAIN_DATASETS) + "\n")
        f.write("Test datasets: " + str(TEST_DATASETS) + "\n")
        f.write("Train age group: " + str(TRAIN_AGE_GROUP) + "\n")
        f.write("Test age group: " + str(TEST_AGE_GROUP) + "\n")
        f.write("Best val EER: " + str(round(best_val_eer, 6)) + "\n")
        f.write("TEST ROC-AUC: " + str(round(test_metrics["roc_auc"], 6)) + "\n")
        f.write("TEST EER: " + str(round(test_metrics["eer"], 6)) + "\n")
        f.write("TEST Rank-1: " + str(round(test_metrics["rank1"], 6)) + "\n")
        f.write("TEST Rank-5: " + str(round(test_metrics["rank5"], 6)) + "\n")

    print("Resultados guardados en:", results_path)


def save_training_history(history):

    history_df = pd.DataFrame(history)

    history_path = os.path.join(OUTPUT_DIR, "training_history.csv")
    history_df.to_csv(history_path, index=False)

    print("Historial guardado en:", history_path)

    return history_df


def save_test_roc_points(test_metrics):

    roc_points_path = os.path.join(OUTPUT_DIR, "test_roc_points.csv")

    roc_df = pd.DataFrame({
        "fpr": test_metrics["fpr"],
        "tpr": test_metrics["tpr"]
    })

    roc_df.to_csv(roc_points_path, index=False)

    print("Puntos ROC guardados en:", roc_points_path)


In [ ]:
def build_experiment_result(best_val_eer, test_metrics, df_train, df_val, df_test):

    experiment_result = {
        "experiment_name": EXPERIMENT_NAME,
        "model_type": BACKBONE_NAME,
        "train_datasets": "+".join(TRAIN_DATASETS),
        "test_datasets": "+".join(TEST_DATASETS),
        "train_age_group": TRAIN_AGE_GROUP,
        "test_age_group": TEST_AGE_GROUP,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "embedding_dim": EMBEDDING_DIM,
        "similarity": "cosine",
        "selection_metric": "val_eer",
        "best_val_eer": float(best_val_eer),
        "test_roc_auc": float(test_metrics["roc_auc"]),
        "test_eer": float(test_metrics["eer"]),
        "test_rank1": float(test_metrics["rank1"]),
        "test_rank5": float(test_metrics["rank5"]),
        "num_train_images": int(len(df_train)),
        "num_val_images": int(len(df_val)),
        "num_test_images": int(len(df_test)),
        "num_train_subjects": int(df_train["subject_id"].nunique()),
        "num_val_subjects": int(df_val["subject_id"].nunique()),
        "num_test_subjects": int(df_test["subject_id"].nunique()),
    }

    return experiment_result


In [ ]:
def update_global_results_csv(experiment_result):

    results_csv = "all_experiments_results.csv"

    if os.path.exists(results_csv):
        results_df = pd.read_csv(results_csv)
        results_df = pd.concat(
            [results_df, pd.DataFrame([experiment_result])],
            ignore_index=True
        )
    else:
        results_df = pd.DataFrame([experiment_result])

    results_df.to_csv(results_csv, index=False)

    print("Resumen global guardado en:", results_csv)


def save_experiment_outputs(history, test_metrics, best_val_eer, df_train, df_val, df_test):

    save_results_txt(
        test_metrics=test_metrics,
        best_val_eer=best_val_eer
    )

    history_df = save_training_history(history)

    save_test_roc_points(test_metrics)

    experiment_result = build_experiment_result(
        best_val_eer=best_val_eer,
        test_metrics=test_metrics,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test
    )

    update_global_results_csv(experiment_result)

    return history_df

## Experimentos base

In [ ]:
#Configuración de parámetros

OUTPUT_DIR = "runs/EICZA_to_UERC"
EXPERIMENT_NAME = "EICZA_to_UERC"

TRAIN_DATASETS = ["EICZA"]
TEST_DATASETS = ["UERC"]

TRAIN_AGE_GROUP = "niño"
TEST_AGE_GROUP = "adulto"

BACKBONE_NAME = "ResNet18"

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-4
NUM_WORKERS = 2
SEED = 42

VAL_RATIO = 0.1

N_GENUINE_PAIRS = 3000
N_IMPOSTOR_PAIRS = 3000

EMBEDDING_DIM = 512

In [ ]:

def main():
    set_seed(SEED)
    make_dir(OUTPUT_DIR)

    device = get_device()
    print("Device:", device)

    df = pd.read_csv(METADATA_CSV)

    df_train, df_val, train_subjects = prepare_train_val_data(df)

    train_loader = create_train_loader(df_train)

    model = create_model(
        num_classes=len(train_subjects),
        device=device
    )

    best_model_path = os.path.join(OUTPUT_DIR, "best_model.pt")

    history, best_val_eer = train_model(
        model=model,
        train_loader=train_loader,
        df_val=df_val,
        device=device,
        best_model_path=best_model_path
    )

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    df_test = prepare_test_data(df)

    test_metrics = evaluate_test_model(
        model=model,
        df_test=df_test,
        device=device
    )

    history_df = save_experiment_outputs(
        history=history,
        test_metrics=test_metrics,
        best_val_eer=best_val_eer,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test
    )

    return history_df


history_df = main()

##Triplet loss

In [ ]:
#Configuración de parámetros

OUTPUT_DIR = "runs/AMI_to_EICZA_triplet"
EXPERIMENT_NAME = "AMI_to_EICZA_triplet"

TRAIN_DATASETS = ["AMI"]
TEST_DATASETS = ["EICZA"]

TRAIN_AGE_GROUP = "adulto"
TEST_AGE_GROUP = "niño"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
NUM_WORKERS = 2
SEED = 42

VAL_RATIO = 0.1

N_GENUINE_PAIRS = 3000
N_IMPOSTOR_PAIRS = 3000

EMBEDDING_DIM = 256
MARGIN = 0.3



class TripletEarDataset(Dataset):
    def __init__(self, df, images_root, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.images_root = images_root
        self.transform = transform

        self.df["subject_id"] = self.df["subject_id"].astype(str)

        self.indices_by_subject = {}
        for idx, row in self.df.iterrows():
            s = row["subject_id"]
            self.indices_by_subject.setdefault(s, []).append(idx)

        self.subjects = list(self.indices_by_subject.keys())
        self.valid_subjects = [s for s in self.subjects if len(self.indices_by_subject[s]) >= 2]

        if len(self.valid_subjects) == 0:
            raise ValueError("No hay sujetos con al menos 2 imágenes para generar tripletes.")

        if len(self.subjects) < 2:
            raise ValueError("Se necesitan al menos 2 sujetos para triplet loss.")

    def __len__(self):
        return len(self.df)

    def _load_image(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.images_root, str(row["image_path"]))
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image

    def __getitem__(self, idx):
        anchor_row = self.df.iloc[idx]
        anchor_subject = anchor_row["subject_id"]

        if anchor_subject not in self.valid_subjects:
            anchor_subject = random.choice(self.valid_subjects)
            idx = random.choice(self.indices_by_subject[anchor_subject])

        anchor_idx = idx

        positive_candidates = self.indices_by_subject[anchor_subject].copy()
        positive_candidates.remove(anchor_idx)
        positive_idx = random.choice(positive_candidates)

        negative_subject = random.choice([s for s in self.subjects if s != anchor_subject])
        negative_idx = random.choice(self.indices_by_subject[negative_subject])

        anchor_img = self._load_image(anchor_idx)
        positive_img = self._load_image(positive_idx)
        negative_img = self._load_image(negative_idx)

        return anchor_img, positive_img, negative_img


class EarInferenceDataset(Dataset):
    def __init__(self, df, images_root, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_root = images_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.images_root, str(row["image_path"]))
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image



class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.embedding(feat)
        emb = F.normalize(emb, p=2, dim=1)
        return emb

@torch.no_grad()
def extract_embeddings_triplet(model, df, images_root, img_size, batch_size, num_workers, device):
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    dataset = EarInferenceDataset(df, images_root, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    embeddings = []

    for x in tqdm(loader, desc="embeddings", leave=False):
        x = x.to(device, non_blocking=True)
        emb = model(x)
        embeddings.append(emb.cpu().numpy())

    embeddings = np.vstack(embeddings)
    embeddings = normalize_embeddings(embeddings)
    return embeddings

@torch.no_grad()
def evaluate_open_set_triplet(model, df_eval, images_root, img_size, batch_size, num_workers, device,
                      n_genuine_pairs=3000, n_impostor_pairs=3000, seed=42):
    if len(df_eval) == 0:
        raise ValueError("El dataframe de evaluación está vacío.")


    embeddings = extract_embeddings_triplet(
        model, df_eval, images_root, img_size, batch_size, num_workers, device
    )
    subject_ids = df_eval["subject_id"].astype(str).values

    y_true, y_score = sample_verification_pairs(
        embeddings, subject_ids, n_genuine_pairs, n_impostor_pairs, seed=seed
    )

    roc_auc = roc_auc_score(y_true, y_score)
    eer, fpr, tpr = compute_eer(y_true, y_score)


    df_gallery, df_probe = split_gallery_probe_by_subject(df_eval, seed=seed)

    if len(df_probe) == 0:
        rank1 = np.nan
        rank5 = np.nan
    else:
        gallery_embeddings = extract_embeddings_triplet(
            model, df_gallery, images_root, img_size, batch_size, num_workers, device
        )
        probe_embeddings = extract_embeddings_triplet(
            model, df_probe, images_root, img_size, batch_size, num_workers, device
        )

        gallery_subject_ids = df_gallery["subject_id"].astype(str).values
        probe_subject_ids = df_probe["subject_id"].astype(str).values

        rank1 = compute_rank_k(
            gallery_embeddings, gallery_subject_ids,
            probe_embeddings, probe_subject_ids,
            k=1
        )

        k5 = min(5, len(np.unique(gallery_subject_ids)))
        rank5 = compute_rank_k(
            gallery_embeddings, gallery_subject_ids,
            probe_embeddings, probe_subject_ids,
            k=k5
        )

    metrics = {
        "roc_auc": float(roc_auc),
        "eer": float(eer),
        "rank1": float(rank1) if not np.isnan(rank1) else np.nan,
        "rank5": float(rank5) if not np.isnan(rank5) else np.nan,
        "fpr": fpr,
        "tpr": tpr,
    }

    return metrics

def train_one_epoch_triplet(model, loader, optimizer, criterion, device):
    model.train()

    all_losses = []

    for anchor, positive, negative in tqdm(loader, desc="train", leave=False):
        anchor = anchor.to(device, non_blocking=True)
        positive = positive.to(device, non_blocking=True)
        negative = negative.to(device, non_blocking=True)

        optimizer.zero_grad()

        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        all_losses.append(loss.item())

    return float(np.mean(all_losses))



In [ ]:


def create_triplet_train_loader(df_train):

    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.RandomRotation(degrees=8),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    train_dataset = TripletEarDataset(
        df=df_train,
        images_root=IMAGES_ROOT,
        transform=train_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True
    )

    return train_loader


def create_triplet_model(device, embedding_dim=EMBEDDING_DIM):

    model = EmbeddingNet(
        embedding_dim=embedding_dim
    ).to(device)

    return model


def initialize_triplet_history():

    history = {
        "epoch": [],
        "train_loss": [],
        "val_roc_auc": [],
        "val_eer": [],
        "val_rank1": [],
        "val_rank5": [],
    }

    return history


def update_triplet_history(history, epoch, train_loss, val_metrics):

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["val_roc_auc"].append(val_metrics["roc_auc"])
    history["val_eer"].append(val_metrics["eer"])
    history["val_rank1"].append(val_metrics["rank1"])
    history["val_rank5"].append(val_metrics["rank5"])


def train_triplet_model(model, train_loader, df_val, device, best_model_path,margin=MARGIN,lr=LR):

    criterion = nn.TripletMarginLoss(margin=margin, p=2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_val_eer = np.inf
    history = initialize_triplet_history()

    print("\n===== TRAINING TRIPLET MODEL =====")
    print("Margin:", margin)
    print("Learning rate:", lr)

    for epoch in range(EPOCHS):
        train_loss = train_one_epoch_triplet(model, train_loader, optimizer, criterion,device)

        val_metrics = evaluate_open_set_triplet(
            model=model,
            df_eval=df_val,
            images_root=IMAGES_ROOT,
            img_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            device=device,
            n_genuine_pairs=N_GENUINE_PAIRS,
            n_impostor_pairs=N_IMPOSTOR_PAIRS,
            seed=SEED
        )

        print(
            f"Epoch {epoch+1:02d} | "
            f"train loss {train_loss:.4f} | "
            f"val AUC {val_metrics['roc_auc']:.4f} | "
            f"val EER {val_metrics['eer']:.4f} | "
            f"val R1 {val_metrics['rank1']:.4f} | "
            f"val R5 {val_metrics['rank5']:.4f}"
        )

        if val_metrics["eer"] < best_val_eer:
            best_val_eer = val_metrics["eer"]
            torch.save(model.state_dict(), best_model_path)
            print("  ✔ Modelo guardado en:", best_model_path)

        update_triplet_history(
            history=history,
            epoch=epoch,
            train_loss=train_loss,
            val_metrics=val_metrics
        )

    print("\nTraining finished.")
    print("Best val EER:", best_val_eer)

    return history, best_val_eer


def save_triplet_results_txt(
    test_metrics,
    best_val_eer,
    experiment_name=EXPERIMENT_NAME,
    output_dir=OUTPUT_DIR,
    embedding_dim=EMBEDDING_DIM,
    margin=MARGIN,
    lr=LR
):

    results_path = os.path.join(output_dir, "results.txt")

    with open(results_path, "w", encoding="utf-8") as f:
        f.write("Experiment name: " + str(experiment_name) + "\n")
        f.write("Model type: resnet18_triplet\n")
        f.write("Train datasets: " + str(TRAIN_DATASETS) + "\n")
        f.write("Test datasets: " + str(TEST_DATASETS) + "\n")
        f.write("Train age group: " + str(TRAIN_AGE_GROUP) + "\n")
        f.write("Test age group: " + str(TEST_AGE_GROUP) + "\n")
        f.write("Embedding dim: " + str(embedding_dim) + "\n")
        f.write("Margin: " + str(margin) + "\n")
        f.write("LR: " + str(lr) + "\n")
        f.write("Best val EER: " + str(round(best_val_eer, 6)) + "\n")
        f.write("TEST ROC-AUC: " + str(round(test_metrics["roc_auc"], 6)) + "\n")
        f.write("TEST EER: " + str(round(test_metrics["eer"], 6)) + "\n")
        f.write("TEST Rank-1: " + str(round(test_metrics["rank1"], 6)) + "\n")
        f.write("TEST Rank-5: " + str(round(test_metrics["rank5"], 6)) + "\n")

    print("Resultados guardados en:", results_path)


def build_triplet_experiment_result(best_val_eer, test_metrics, df_train, df_val, df_test, experiment_name=EXPERIMENT_NAME, embedding_dim=EMBEDDING_DIM, margin=MARGIN,lr=LR):

    experiment_result = {
        "experiment_name": experiment_name,
        "model_type": "resnet18_triplet",
        "train_datasets": "+".join(TRAIN_DATASETS),
        "test_datasets": "+".join(TEST_DATASETS),
        "train_age_group": TRAIN_AGE_GROUP,
        "test_age_group": TEST_AGE_GROUP,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,

        "lr": lr,
        "embedding_dim": embedding_dim,
        "margin": margin,

        "selection_metric": "val_eer",
        "best_val_eer": float(best_val_eer),
        "test_roc_auc": float(test_metrics["roc_auc"]),
        "test_eer": float(test_metrics["eer"]),
        "test_rank1": float(test_metrics["rank1"]),
        "test_rank5": float(test_metrics["rank5"]),

        "num_train_images": int(len(df_train)),
        "num_val_images": int(len(df_val)),
        "num_test_images": int(len(df_test)),
        "num_train_subjects": int(df_train["subject_id"].nunique()),
        "num_val_subjects": int(df_val["subject_id"].nunique()),
        "num_test_subjects": int(df_test["subject_id"].nunique()),
    }

    return experiment_result


def update_global_results_csv_triplet(experiment_result):

    results_csv = "all_experiments_results.csv"

    if os.path.exists(results_csv):
        results_df = pd.read_csv(results_csv)
        results_df = pd.concat(
            [results_df, pd.DataFrame([experiment_result])],
            ignore_index=True
        )
    else:
        results_df = pd.DataFrame([experiment_result])

    results_df.to_csv(results_csv, index=False)

    print("Resumen global guardado en:", results_csv)


def save_triplet_experiment_outputs(
    history,
    test_metrics,
    best_val_eer,
    df_train,
    df_val,
    df_test,
    experiment_name=EXPERIMENT_NAME,
    output_dir=OUTPUT_DIR,
    embedding_dim=EMBEDDING_DIM,
    margin=MARGIN,
    lr=LR
):

    history_df = pd.DataFrame(history)

    history_path = os.path.join(output_dir, "training_history.csv")
    history_df.to_csv(history_path, index=False)
    print("Historial guardado en:", history_path)

    save_triplet_results_txt(
        test_metrics=test_metrics,
        best_val_eer=best_val_eer,
        experiment_name=experiment_name,
        output_dir=output_dir,
        embedding_dim=embedding_dim,
        margin=margin,
        lr=lr
    )

    roc_points_path = os.path.join(output_dir, "test_roc_points.csv")
    pd.DataFrame({
        "fpr": test_metrics["fpr"],
        "tpr": test_metrics["tpr"]
    }).to_csv(roc_points_path, index=False)
    print("Puntos ROC guardados en:", roc_points_path)

    experiment_result = build_triplet_experiment_result(
        best_val_eer=best_val_eer,
        test_metrics=test_metrics,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        experiment_name=experiment_name,
        embedding_dim=embedding_dim,
        margin=margin,
        lr=lr
    )

    update_global_results_csv_triplet(experiment_result)

    return history_df, experiment_result

def evaluate_test_model_triplet(model, df_test, device):

    test_metrics = evaluate_open_set_triplet(
        model=model,
        df_eval=df_test,
        images_root=IMAGES_ROOT,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
        n_genuine_pairs=N_GENUINE_PAIRS,
        n_impostor_pairs=N_IMPOSTOR_PAIRS,
        seed=SEED
    )

    print("\n===== RESULTADOS TEST TRIPLET =====")
    print("ROC-AUC:", round(test_metrics["roc_auc"], 4))
    print("EER:", round(test_metrics["eer"], 4))
    print("Rank-1:", round(test_metrics["rank1"], 4))
    print("Rank-5:", round(test_metrics["rank5"], 4))

    return test_metrics

In [ ]:


def main_triplet(
    embedding_dim=EMBEDDING_DIM,
    margin=MARGIN,
    lr=LR,
    experiment_name=None,
    output_dir=None
):

    set_seed(SEED)

    if experiment_name is None:
        experiment_name = EXPERIMENT_NAME

    if output_dir is None:
        output_dir = OUTPUT_DIR

    make_dir(output_dir)

    device = get_device()
    print("Device:", device)
    print("Experiment:", experiment_name)
    print("Embedding dim:", embedding_dim)
    print("Margin:", margin)
    print("LR:", lr)

    df = pd.read_csv(METADATA_CSV)

    df_train, df_val, _ = prepare_train_val_data(df)

    train_loader = create_triplet_train_loader(df_train)

    model = create_triplet_model(
        device=device,
        embedding_dim=embedding_dim
    )

    best_model_path = os.path.join(output_dir, "best_model.pt")

    history, best_val_eer = train_triplet_model(
        model=model,
        train_loader=train_loader,
        df_val=df_val,
        device=device,
        best_model_path=best_model_path,
        margin=margin,
        lr=lr
    )

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    df_test = prepare_test_data(df)

    test_metrics = evaluate_test_model_triplet(
        model=model,
        df_test=df_test,
        device=device
    )

    history_df, experiment_result = save_triplet_experiment_outputs(
        history=history,
        test_metrics=test_metrics,
        best_val_eer=best_val_eer,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        experiment_name=experiment_name,
        output_dir=output_dir,
        embedding_dim=embedding_dim,
        margin=margin,
        lr=lr
    )

    return history_df, experiment_result

history_df, experiment_result = main_triplet()


##Tunning Hiperparámetros

In [ ]:
#Configuración de parámetros

OUTPUT_DIR = "runs/AMI_to_EICZA_triplet_tuning"
EXPERIMENT_NAME = "AMI_to_EICZA_triplet_tuning"

TRAIN_DATASETS = ["AMI"]
TEST_DATASETS = ["EICZA"]

TRAIN_AGE_GROUP = "adulto"
TEST_AGE_GROUP = "niño"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
NUM_WORKERS = 2
SEED = 42

VAL_RATIO = 0.1

N_GENUINE_PAIRS = 3000
N_IMPOSTOR_PAIRS = 3000





def run_hyperparameter_tuning():

    param_grid = [
        # embedding_dim, margin, lr
        (128, 0.2, 1e-4),
        (256, 0.2, 1e-4),
        (512, 0.2, 1e-4),
    ]

    tuning_results = []

    for i, (embedding_dim, margin, lr) in enumerate(param_grid, start=1):

        experiment_name = (
            f"{EXPERIMENT_NAME}"
            f"_emb{embedding_dim}"
            f"_m{margin}"
            f"_lr{lr}"
        )

        output_dir = os.path.join(
            OUTPUT_DIR,
            experiment_name
        )

        print("\n" + "=" * 80)
        print(f"TUNING {i}/{len(param_grid)}")
        print("Experiment:", experiment_name)
        print("Output dir:", output_dir)
        print("Embedding dim:", embedding_dim)
        print("Margin:", margin)
        print("LR:", lr)
        print("=" * 80)

        _, experiment_result = main_triplet(
            embedding_dim=embedding_dim,
            margin=margin,
            lr=lr,
            experiment_name=experiment_name,
            output_dir=output_dir
        )

        tuning_results.append(experiment_result)

    tuning_df = pd.DataFrame(tuning_results)
    tuning_df = tuning_df.sort_values("test_eer", ascending=True)

    make_dir(OUTPUT_DIR)

    tuning_summary_path = os.path.join(
        OUTPUT_DIR,
        "hyperparameter_tuning_summary.csv"
    )

    tuning_df.to_csv(tuning_summary_path, index=False)

    print("\n===== RESUMEN TUNING =====")
    print(
        tuning_df[
            [
                "experiment_name",
                "embedding_dim",
                "margin",
                "lr",
                "best_val_eer",
                "test_roc_auc",
                "test_eer",
                "test_rank1",
                "test_rank5",
            ]
        ]
    )

    print("\nResumen tuning guardado en:", tuning_summary_path)

    return tuning_df


tuning_df = run_hyperparameter_tuning()



##Comparación de backbones

In [ ]:


# Lista de todos los modelos de keras
model_fns = {
"MobileNet": keras.applications.MobileNet,
"MobileNetV2": keras.applications.MobileNetV2,
"MobileNetV3Small": keras.applications.MobileNetV3Small,
"MobileNetV3Large": keras.applications.MobileNetV3Large,
"EfficientNetB0": keras.applications.EfficientNetB0,
"EfficientNetB1": keras.applications.EfficientNetB1,
"EfficientNetB2": keras.applications.EfficientNetB2,
"ResNet50": keras.applications.ResNet50,
"DenseNet121": keras.applications.DenseNet121,
"VGG16": keras.applications.VGG16,
}

resultados = []

for nombre, fn in model_fns.items():
  model = fn(weights="imagenet", include_top=False)

  params = model.count_params()

  with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, f"{nombre}.keras")
    model.save(path)
    size_bytes = os.path.getsize(path)

  resultados.append({
  "modelo": nombre,
  "params": params,
  "size_mb": round(size_bytes / (1024 * 1024), 2),
  })

ordenados_params = sorted(resultados, key=lambda x: x["params"])

print("Ordenados por parámetros:")
for r in ordenados_params:
  print(f"{r['modelo']:20s} params={r['params']:,} size={r['size_mb']} MB")

ordenados_size = sorted(resultados, key=lambda x: x["size_mb"])

print("\nOrdenados por tamaño en disco:")
for r in ordenados_size:
  print(f"{r['modelo']:20s} params={r['params']:,} size={r['size_mb']} MB")

Backbones

In [ ]:
#Configuración de parámetros


BASE_OUTPUT_DIR = "runs/backbone_comparison_multi"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-4
NUM_WORKERS = 2
SEED = 42

VAL_RATIO = 0.1

N_GENUINE_PAIRS = 3000
N_IMPOSTOR_PAIRS = 3000

EMBEDDING_DIM = 512

BACKBONE_NAME = "resnet18"
# Opciones:
# "resnet18"
# "resnet50"
# "mobilenet_v2"
# "mobilenet_v3_small"
# "mobilenet_v3_large"
# "efficientnet_b0"
# "efficientnet_b1"
# "densenet121"

EXPERIMENTS = [
    {
        "experiment_name": "AMI_to_EICZA",
        "train_datasets": ["AMI"],
        "test_datasets": ["EICZA"],
        "train_age_group": "adulto",
        "test_age_group": "niño",
    },
    {
        "experiment_name": "BIPLab_to_EICZA",
        "train_datasets": ["BIPLab"],
        "test_datasets": ["EICZA"],
        "train_age_group": "adulto",
        "test_age_group": "niño",
    },
    {
        "experiment_name": "UERC_to_EICZA",
        "train_datasets": ["UERC"],
        "test_datasets": ["EICZA"],
        "train_age_group": "adulto",
        "test_age_group": "niño",
    },
]




def get_experiment_output_dir(experiment_name, backbone_name):
    folder_name = f"{experiment_name}__{backbone_name}"
    return os.path.join(BASE_OUTPUT_DIR, folder_name)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_backbone(backbone_name):
    backbone_name = backbone_name.lower()

    if backbone_name == "resnet18":
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        return backbone, in_features

    elif backbone_name == "resnet50":
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        return backbone, in_features

    elif backbone_name == "mobilenet_v2":
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        return backbone, in_features

    elif backbone_name == "mobilenet_v3_small":
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()
        return backbone, in_features

    elif backbone_name == "mobilenet_v3_large":
        backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        in_features = backbone.classifier[0].in_features
        backbone.classifier = nn.Identity()
        return backbone, in_features

    elif backbone_name == "efficientnet_b0":
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        return backbone, in_features

    elif backbone_name == "efficientnet_b1":
        backbone = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        return backbone, in_features

    elif backbone_name == "densenet121":
        backbone = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_features = backbone.classifier.in_features
        backbone.classifier = nn.Identity()
        return backbone, in_features

    else:
        raise ValueError(f"Backbone no soportado: {backbone_name}")


class EmbeddingClassifierBackbone(nn.Module):
    def __init__(self, num_classes, backbone_name, embedding_dim=512):
        super().__init__()

        backbone, in_features = build_backbone(backbone_name)

        self.backbone = backbone
        self.backbone_name = backbone_name

        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embedding_dim, embedding_dim)
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        emb = self.embedding(features)
        logits = self.classifier(emb)
        return logits, emb


def create_model_for_backbone_comparison(num_classes, device):

    model = EmbeddingClassifierBackbone(
        num_classes=num_classes,
        backbone_name=BACKBONE_NAME,
        embedding_dim=EMBEDDING_DIM
    ).to(device)

    return model



def configure_experiment_globals(experiment_cfg):

    global EXPERIMENT_NAME
    global TRAIN_DATASETS
    global TEST_DATASETS
    global TRAIN_AGE_GROUP
    global TEST_AGE_GROUP
    global OUTPUT_DIR

    EXPERIMENT_NAME = experiment_cfg["experiment_name"]
    TRAIN_DATASETS = experiment_cfg["train_datasets"]
    TEST_DATASETS = experiment_cfg["test_datasets"]
    TRAIN_AGE_GROUP = experiment_cfg["train_age_group"]
    TEST_AGE_GROUP = experiment_cfg["test_age_group"]

    OUTPUT_DIR = get_experiment_output_dir(EXPERIMENT_NAME, BACKBONE_NAME)
    make_dir(OUTPUT_DIR)

    return OUTPUT_DIR




def compute_cmc_curve(gallery_embeddings, gallery_subject_ids,
                      probe_embeddings, probe_subject_ids, max_rank=None):

    if len(probe_embeddings) == 0:
        return {}

    num_gallery = len(gallery_subject_ids)

    if max_rank is None:
        max_rank = num_gallery

    max_rank = min(max_rank, num_gallery)

    sims = probe_embeddings @ gallery_embeddings.T
    sorted_idx = np.argsort(-sims, axis=1)
    ranked_subjects = gallery_subject_ids[sorted_idx]

    cmc = {}
    n_probe = len(probe_subject_ids)

    for k in range(1, max_rank + 1):
        correct = 0

        for i in range(n_probe):
            if probe_subject_ids[i] in ranked_subjects[i, :k]:
                correct += 1

        cmc[k] = float(correct / n_probe)

    return cmc


@torch.no_grad()
def compute_cmc_for_model(model, df_eval, images_root, img_size,
                          batch_size, num_workers, device, seed=42):

    if len(df_eval) == 0:
        raise ValueError("El dataframe de evaluación está vacío.")

    df_gallery, df_probe = split_gallery_probe_by_subject(df_eval, seed=seed)

    if len(df_probe) == 0:
        return {}

    gallery_embeddings = extract_embeddings(
        model,
        df_gallery,
        images_root,
        img_size,
        batch_size,
        num_workers,
        device
    )

    probe_embeddings = extract_embeddings(
        model,
        df_probe,
        images_root,
        img_size,
        batch_size,
        num_workers,
        device
    )

    gallery_subject_ids = df_gallery["subject_id"].astype(str).values
    probe_subject_ids = df_probe["subject_id"].astype(str).values

    cmc_curve = compute_cmc_curve(
        gallery_embeddings=gallery_embeddings,
        gallery_subject_ids=gallery_subject_ids,
        probe_embeddings=probe_embeddings,
        probe_subject_ids=probe_subject_ids,
        max_rank=len(gallery_subject_ids)
    )

    return cmc_curve


def save_comparative_cmc_curve(cmc_results, output_path, title):

    if len(cmc_results) == 0:
        return

    plt.figure(figsize=(9, 6))

    for exp_name, cmc_dict in cmc_results.items():
        if len(cmc_dict) == 0:
            continue

        ranks = list(cmc_dict.keys())
        values = list(cmc_dict.values())

        plt.plot(ranks, values, linewidth=2, label=exp_name)

    plt.xlabel("Rank")
    plt.ylabel("Identification Rate")
    plt.title(title)
    plt.ylim(0.0, 1.02)

    max_rank = 1

    for cmc_dict in cmc_results.values():
        if len(cmc_dict) > 0:
            max_rank = max(max_rank, max(cmc_dict.keys()))

    plt.xlim(1, max_rank)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


def save_comparative_cmc_csv(comparative_cmc, backbone_name, base_output_dir):

    comparative_rows = []

    for exp_name, cmc_dict in comparative_cmc.items():
        for rank, identification_rate in cmc_dict.items():
            comparative_rows.append({
                "experiment_name": exp_name,
                "backbone": backbone_name,
                "rank": rank,
                "identification_rate": identification_rate
            })

    comparative_cmc_csv = os.path.join(
        base_output_dir,
        f"comparative_cmc__{backbone_name}.csv"
    )

    comparative_df = pd.DataFrame(comparative_rows)
    comparative_df.to_csv(comparative_cmc_csv, index=False)

    print("CSV CMC comparativo guardado en:", comparative_cmc_csv)

    return comparative_df



def build_experiment_result(best_val_eer, test_metrics, df_train, df_val, df_test):

    experiment_result = {
        "experiment_name": EXPERIMENT_NAME,
        "model_type": BACKBONE_NAME,
        "train_datasets": "+".join(TRAIN_DATASETS),
        "test_datasets": "+".join(TEST_DATASETS),
        "train_age_group": TRAIN_AGE_GROUP,
        "test_age_group": TEST_AGE_GROUP,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "embedding_dim": EMBEDDING_DIM,
        "similarity": "cosine",
        "selection_metric": "val_eer",
        "best_val_eer": float(best_val_eer),
        "test_roc_auc": float(test_metrics["roc_auc"]),
        "test_eer": float(test_metrics["eer"]),
        "test_rank1": float(test_metrics["rank1"]),
        "test_rank5": float(test_metrics["rank5"]),
        "num_train_images": int(len(df_train)),
        "num_val_images": int(len(df_val)),
        "num_test_images": int(len(df_test)),
        "num_train_subjects": int(df_train["subject_id"].nunique()),
        "num_val_subjects": int(df_val["subject_id"].nunique()),
        "num_test_subjects": int(df_test["subject_id"].nunique()),
    }

    return experiment_result


def update_global_results_csv_for_backbone_comparison(results_df):

    results_csv = "all_experiments_results.csv"

    if os.path.exists(results_csv):
        previous_df = pd.read_csv(results_csv)
        results_df = pd.concat(
            [previous_df, results_df],
            ignore_index=True
        )

    results_df.to_csv(results_csv, index=False)

    print("Resumen global guardado en:", results_csv)

    return results_df


def save_experiment_outputs_for_backbone_comparison(
    history,
    test_metrics,
    best_val_eer,
    df_train,
    df_val,
    df_test
):

    save_results_txt(
        test_metrics=test_metrics,
        best_val_eer=best_val_eer
    )

    history_df = save_training_history(history)

    save_test_roc_points(test_metrics)

    experiment_result = build_experiment_result(
        best_val_eer=best_val_eer,
        test_metrics=test_metrics,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test
    )

    return experiment_result, history_df




def run_single_experiment(df, experiment_cfg, device):
    output_dir = configure_experiment_globals(experiment_cfg)

    print("\n" + "=" * 80)
    print(f"Running experiment: {EXPERIMENT_NAME} | backbone={BACKBONE_NAME}")
    print("=" * 80)


    df_train, df_val, train_subjects = prepare_train_val_data(df)


    train_loader = create_train_loader(df_train)


    model = create_model_for_backbone_comparison(
        num_classes=len(train_subjects),
        device=device
    )

    num_params = count_parameters(model)
    print("Trainable params:", num_params)

    best_model_path = os.path.join(output_dir, "best_model.pt")


    history, best_val_eer = train_model(
        model=model,
        train_loader=train_loader,
        df_val=df_val,
        device=device,
        best_model_path=best_model_path
    )


    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    df_test = prepare_test_data(df)

    test_metrics = evaluate_test_model(
        model=model,
        df_test=df_test,
        device=device
    )


    cmc_curve = compute_cmc_for_model(
        model=model,
        df_eval=df_test,
        images_root=IMAGES_ROOT,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
        seed=SEED
    )


    experiment_result, _ = save_experiment_outputs_for_backbone_comparison(
        history=history,
        test_metrics=test_metrics,
        best_val_eer=best_val_eer,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test
    )

    return experiment_result, cmc_curve




def main_backbone():
    set_seed(SEED)
    make_dir(BASE_OUTPUT_DIR)

    device = get_device()
    print("Device:", device)
    print("Backbone:", BACKBONE_NAME)

    df = pd.read_csv(METADATA_CSV)

    all_results = []
    comparative_cmc = {}

    for experiment_cfg in EXPERIMENTS:
        experiment_result, cmc_curve = run_single_experiment(
            df=df,
            experiment_cfg=experiment_cfg,
            device=device
        )

        all_results.append(experiment_result)
        comparative_cmc[experiment_cfg["experiment_name"]] = cmc_curve


    new_results_df = pd.DataFrame(all_results)

    results_df = update_global_results_csv_for_backbone_comparison(
        results_df=new_results_df
    )


    comparative_cmc_plot = os.path.join(
        BASE_OUTPUT_DIR,
        f"comparative_cmc__{BACKBONE_NAME}.png"
    )

    save_comparative_cmc_curve(
        cmc_results=comparative_cmc,
        output_path=comparative_cmc_plot,
        title=f"Comparative CMC - {BACKBONE_NAME}"
    )

    print("Gráfica CMC comparativa guardada en:", comparative_cmc_plot)


    save_comparative_cmc_csv(
        comparative_cmc=comparative_cmc,
        backbone_name=BACKBONE_NAME,
        base_output_dir=BASE_OUTPUT_DIR
    )

    return results_df


results_df = main_backbone()


##Fusión de embeddings

In [ ]:
# #Configuración de parámetros


BASE_OUTPUT_DIR = "runs/embedding_fusion"

EXPERIMENT_NAME = "BIPLab_to_EICZA_resnet18_mobilenetv3large_simple_fusion"

TRAIN_DATASETS = ["BIPLab"]
TRAIN_AGE_GROUP = "adulto"

TEST_DATASETS = ["EICZA"]
TEST_AGE_GROUP = "niño"

BACKBONE_1 = "resnet18"
BACKBONE_2 = "mobilenet_v3_large"

CHECKPOINT_1 = "Datos_TFG_Rafael/best_model_Resnet.pt"
CHECKPOINT_2 = "Datos_TFG_Rafael/best_model_mobilenet.pt"

FUSION_NAME = f"{BACKBONE_1}+{BACKBONE_2}_simple_fusion"
FUSED_EMBEDDING_DIM = EMBEDDING_DIM * 2

OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, EXPERIMENT_NAME)

In [ ]:
def load_model_for_embeddings(checkpoint_path, backbone_name, num_classes, device):

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"No existe el checkpoint: {checkpoint_path}")

    model = EmbeddingClassifierBackbone(
        num_classes=num_classes,
        backbone_name=backbone_name,
        embedding_dim=EMBEDDING_DIM
    ).to(device)

    state_dict = torch.load(checkpoint_path, map_location=device)

    filtered_state_dict = {
        k: v for k, v in state_dict.items()
        if not k.startswith("classifier.")
    }

    model.load_state_dict(filtered_state_dict, strict=False)
    model.eval()

    print(f"\nModelo cargado: {backbone_name}")
    print(f"Checkpoint: {checkpoint_path}")

    return model

In [ ]:
@torch.no_grad()
def extract_fused_embeddings(model_1, model_2, df, device):

    emb_1 = extract_embeddings(
        model=model_1,
        df=df,
        images_root=IMAGES_ROOT,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device
    )

    emb_2 = extract_embeddings(
        model=model_2,
        df=df,
        images_root=IMAGES_ROOT,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device
    )

    emb_1 = normalize_embeddings(emb_1)
    emb_2 = normalize_embeddings(emb_2)

    fused_embeddings = np.concatenate([emb_1, emb_2], axis=1)
    fused_embeddings = normalize_embeddings(fused_embeddings)

    return fused_embeddings

In [ ]:
def evaluate_verification_from_embeddings(embeddings, df_eval):
    subject_ids = df_eval["subject_id"].astype(str).values

    y_true, y_score = sample_verification_pairs(
        embeddings=embeddings,
        subject_ids=subject_ids,
        n_genuine=N_GENUINE_PAIRS,
        n_impostor=N_IMPOSTOR_PAIRS,
        seed=SEED
    )

    roc_auc = roc_auc_score(y_true, y_score)
    eer, fpr, tpr = compute_eer(y_true, y_score)

    return {
        "roc_auc": float(roc_auc),
        "eer": float(eer),
        "fpr": fpr,
        "tpr": tpr
    }


def evaluate_from_embeddings(
    model_name,
    test_embeddings,
    gallery_embeddings,
    probe_embeddings,
    df_test,
    gallery_subject_ids,
    probe_subject_ids
):
    verification_metrics = evaluate_verification_from_embeddings(
        embeddings=test_embeddings,
        df_eval=df_test
    )

    rank1 = compute_rank_k(
        gallery_embeddings=gallery_embeddings,
        gallery_subject_ids=gallery_subject_ids,
        probe_embeddings=probe_embeddings,
        probe_subject_ids=probe_subject_ids,
        k=1
    )

    k5 = min(5, len(np.unique(gallery_subject_ids)))

    rank5 = compute_rank_k(
        gallery_embeddings=gallery_embeddings,
        gallery_subject_ids=gallery_subject_ids,
        probe_embeddings=probe_embeddings,
        probe_subject_ids=probe_subject_ids,
        k=k5
    )

    cmc_curve = compute_cmc_curve(
        gallery_embeddings=gallery_embeddings,
        gallery_subject_ids=gallery_subject_ids,
        probe_embeddings=probe_embeddings,
        probe_subject_ids=probe_subject_ids,
        max_rank=len(gallery_subject_ids)
    )

    return {
        "model_name": model_name,
        "roc_auc": verification_metrics["roc_auc"],
        "eer": verification_metrics["eer"],
        "rank1": float(rank1),
        "rank5": float(rank5),
        "fpr": verification_metrics["fpr"],
        "tpr": verification_metrics["tpr"],
        "cmc_curve": cmc_curve
    }

In [ ]:
def build_results_table(results):
    rows = []

    for result in results:
        rows.append({
            "model": result["model_name"],
            "roc_auc": result["roc_auc"],
            "eer": result["eer"],
            "rank1": result["rank1"],
            "rank5": result["rank5"]
        })

    df_results = pd.DataFrame(rows)

    print("\n===== RESULTADOS COMPARATIVOS =====")
    print(df_results.round(4).to_string(index=False))

    return df_results


def build_simple_fusion_result_row(fusion_result, df_train, df_val, df_test):
    experiment_result = {
        "experiment_name": EXPERIMENT_NAME,
        "model_type": "simple_embedding_fusion",
        "backbone": FUSION_NAME,
        "train_datasets": "+".join(TRAIN_DATASETS),
        "test_datasets": "+".join(TEST_DATASETS),
        "train_age_group": TRAIN_AGE_GROUP,
        "test_age_group": TEST_AGE_GROUP,
        "epochs": 0,
        "batch_size": BATCH_SIZE,
        "lr": 0,
        "embedding_dim": FUSED_EMBEDDING_DIM,
        "similarity": "cosine",
        "selection_metric": "none",
        "best_val_eer": np.nan,
        "test_roc_auc": float(fusion_result["roc_auc"]),
        "test_eer": float(fusion_result["eer"]),
        "test_rank1": float(fusion_result["rank1"]),
        "test_rank5": float(fusion_result["rank5"]),
        "num_train_images": int(len(df_train)),
        "num_val_images": int(len(df_val)),
        "num_test_images": int(len(df_test)),
        "num_train_subjects": int(df_train["subject_id"].nunique()),
        "num_val_subjects": int(df_val["subject_id"].nunique()),
        "num_test_subjects": int(df_test["subject_id"].nunique()),
    }

    return experiment_result

In [ ]:
def main_fusion():
    set_seed(SEED)
    make_dir(OUTPUT_DIR)

    device = get_device()
    print("Device:", device)

    df = pd.read_csv(METADATA_CSV)

    df_train, df_val, train_subjects = prepare_train_val_data(df)
    num_classes = len(train_subjects)

    df_test = prepare_test_data(df)

    df_gallery, df_probe = split_gallery_probe_by_subject(df_test, seed=SEED)

    if len(df_probe) == 0:
        raise ValueError("No hay imágenes probe. Se necesitan sujetos con al menos 2 imágenes.")

    gallery_subject_ids = df_gallery["subject_id"].astype(str).values
    probe_subject_ids = df_probe["subject_id"].astype(str).values

    print("\n===== GALLERY / PROBE SPLIT =====")
    print("Gallery subjects:", df_gallery["subject_id"].nunique())
    print("Gallery images:", len(df_gallery))
    print("Probe subjects:", df_probe["subject_id"].nunique())
    print("Probe images:", len(df_probe))


    model_1 = load_model_for_embeddings(
        checkpoint_path=CHECKPOINT_1,
        backbone_name=BACKBONE_1,
        num_classes=num_classes,
        device=device
    )

    model_2 = load_model_for_embeddings(
        checkpoint_path=CHECKPOINT_2,
        backbone_name=BACKBONE_2,
        num_classes=num_classes,
        device=device
    )


    print("\n===== EXTRACTING INDIVIDUAL EMBEDDINGS =====")

    model_1_test_embeddings = normalize_embeddings(
        extract_embeddings(model_1, df_test, IMAGES_ROOT, IMG_SIZE, BATCH_SIZE, NUM_WORKERS, device)
    )
    model_1_gallery_embeddings = normalize_embeddings(
        extract_embeddings(model_1, df_gallery, IMAGES_ROOT, IMG_SIZE, BATCH_SIZE, NUM_WORKERS, device)
    )
    model_1_probe_embeddings = normalize_embeddings(
        extract_embeddings(model_1, df_probe, IMAGES_ROOT, IMG_SIZE, BATCH_SIZE, NUM_WORKERS, device)
    )

    model_2_test_embeddings = normalize_embeddings(
        extract_embeddings(model_2, df_test, IMAGES_ROOT, IMG_SIZE, BATCH_SIZE, NUM_WORKERS, device)
    )
    model_2_gallery_embeddings = normalize_embeddings(
        extract_embeddings(model_2, df_gallery, IMAGES_ROOT, IMG_SIZE, BATCH_SIZE, NUM_WORKERS, device)
    )
    model_2_probe_embeddings = normalize_embeddings(
        extract_embeddings(model_2, df_probe, IMAGES_ROOT, IMG_SIZE, BATCH_SIZE, NUM_WORKERS, device)
    )


    print("\n===== SIMPLE EMBEDDING FUSION =====")

    fusion_test_embeddings = normalize_embeddings(
        np.concatenate([model_1_test_embeddings, model_2_test_embeddings], axis=1)
    )

    fusion_gallery_embeddings = normalize_embeddings(
        np.concatenate([model_1_gallery_embeddings, model_2_gallery_embeddings], axis=1)
    )

    fusion_probe_embeddings = normalize_embeddings(
        np.concatenate([model_1_probe_embeddings, model_2_probe_embeddings], axis=1)
    )

    model_1_results = evaluate_from_embeddings(
        model_name=BACKBONE_1,
        test_embeddings=model_1_test_embeddings,
        gallery_embeddings=model_1_gallery_embeddings,
        probe_embeddings=model_1_probe_embeddings,
        df_test=df_test,
        gallery_subject_ids=gallery_subject_ids,
        probe_subject_ids=probe_subject_ids
    )

    model_2_results = evaluate_from_embeddings(
        model_name=BACKBONE_2,
        test_embeddings=model_2_test_embeddings,
        gallery_embeddings=model_2_gallery_embeddings,
        probe_embeddings=model_2_probe_embeddings,
        df_test=df_test,
        gallery_subject_ids=gallery_subject_ids,
        probe_subject_ids=probe_subject_ids
    )

    fusion_results = evaluate_from_embeddings(
        model_name=FUSION_NAME,
        test_embeddings=fusion_test_embeddings,
        gallery_embeddings=fusion_gallery_embeddings,
        probe_embeddings=fusion_probe_embeddings,
        df_test=df_test,
        gallery_subject_ids=gallery_subject_ids,
        probe_subject_ids=probe_subject_ids
    )

    results = [
        model_1_results,
        model_2_results,
        fusion_results
    ]


    df_results = build_results_table(results)

    summary_csv_path = os.path.join(
        OUTPUT_DIR,
        "simple_fusion_summary.csv"
    )

    df_results.to_csv(summary_csv_path, index=False)
    print("Resumen comparativo guardado en:", summary_csv_path)


    comparative_cmc = {
        result["model_name"]: result["cmc_curve"]
        for result in results
    }

    comparative_cmc_plot = os.path.join(
        OUTPUT_DIR,
        "comparative_cmc_simple_fusion.png"
    )

    save_comparative_cmc_curve(
        cmc_results=comparative_cmc,
        output_path=comparative_cmc_plot,
        title="Comparative CMC - Simple Embedding Fusion"
    )

    print("Gráfica CMC comparativa guardada en:", comparative_cmc_plot)


    save_comparative_cmc_csv(
        comparative_cmc=comparative_cmc,
        backbone_name="simple_fusion",
        base_output_dir=OUTPUT_DIR
    )

    experiment_result = build_simple_fusion_result_row(
        fusion_result=fusion_results,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test
    )

    new_results_df = pd.DataFrame([experiment_result])

    results_df = update_global_results_csv_for_backbone_comparison(
        results_df=new_results_df
    )

    print("\n===== ARCHIVOS GUARDADOS =====")
    print("Resumen CSV:", summary_csv_path)
    print("CMC plot:", comparative_cmc_plot)
    print("Global CSV: all_experiments_results.csv")

    return df_results



results_fusion_df = main_fusion()
